# 🎬 TC Dataset

## Objective
Build a reusable Tom Cruise reference dataset for downstream agent evaluation, analytics, and dashboarding.

## Story Flow
1. Collect reference content from Wikipedia search results.
2. Optionally enrich with YouTube search results when an API key is available.
3. Normalize all records into a common schema.
4. Ingest any new landed CSV or JSON files from the workspace landing folder.
5. Persist curated tables in Unity Catalog for reuse by the evaluation notebook and dashboards.

## Output Assets
* Curated dataset table
* Source-level raw table
* Ingestion metrics table
* Workspace landing folder for new files: `/Users/yogesh.ghorpade@databricks.com/tc_landing`


## 🏗️ Solution Architecture

`Wikipedia Search API` + `YouTube Search API (optional)` + `New landed files` → `Normalization layer` → `Unity Catalog raw source table` → `Unity Catalog curated Tom Cruise dataset`

### Design Notes
* Wikipedia enrichment is fully automated with the public MediaWiki API.
* YouTube enrichment is parameterized so the notebook works immediately and becomes richer when a YouTube API key is supplied later.
* New incoming files are expected in the workspace folder created for this project.
* The downstream evaluation notebook reads the curated table if it exists, but it can still run without it.


In [0]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
import mlflow

TARGET_CATALOG = "lakemeter_catalog"
TARGET_SCHEMA = "labuser3"
RAW_SOURCE_TABLE = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.tc_dataset_sources"
CURATED_TABLE = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.tc_dataset"
INGEST_METRICS_TABLE = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.tc_dataset_ingest_metrics"
LANDING_DIR = Path("/Workspace/Users/yogesh.ghorpade@databricks.com/tc_landing")
WIKI_QUERY = "Tom Cruise"
YOUTUBE_API_KEY = os.environ.get("YOUTUBE_API_KEY")

mlflow.set_experiment("/Users/yogesh.ghorpade@databricks.com/TC Dataset")

print(f"Target curated table : {CURATED_TABLE}")
print(f"Landing directory    : {LANDING_DIR}")
print(f"YouTube enabled      : {bool(YOUTUBE_API_KEY)}")


Target curated table : lakemeter_catalog.labuser3.tc_dataset
Landing directory    : /Workspace/Users/yogesh.ghorpade@databricks.com/tc_landing
YouTube enabled      : False


In [0]:
def wikipedia_search(query: str, limit: int = 10) -> pd.DataFrame:
    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query,
        "srlimit": limit,
        "utf8": 1,
    }
    response = requests.get("https://en.wikipedia.org/w/api.php", params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    rows = []
    for item in payload.get("query", {}).get("search", []):
        title = item.get("title")
        page_id = item.get("pageid")
        rows.append(
            {
                "source_system": "wikipedia",
                "query_text": query,
                "record_id": str(page_id),
                "title": title,
                "url": f"https://en.wikipedia.org/?curid={page_id}",
                "description": item.get("snippet", "").replace("<span class=\"searchmatch\">", "").replace("</span>", ""),
                "published_at": None,
                "source_payload": json.dumps(item, ensure_ascii=False),
            }
        )
    return pd.DataFrame(rows)


def youtube_search(query: str, max_results: int = 10, api_key: str | None = None) -> pd.DataFrame:
    if not api_key:
        return pd.DataFrame(
            [
                {
                    "source_system": "youtube",
                    "query_text": query,
                    "record_id": "api_key_required",
                    "title": "YouTube enrichment skipped",
                    "url": None,
                    "description": "Set the YOUTUBE_API_KEY environment variable to enable YouTube search ingestion.",
                    "published_at": None,
                    "source_payload": json.dumps({"status": "skipped", "reason": "missing_api_key"}),
                }
            ]
        )

    params = {
        "part": "snippet",
        "q": query,
        "type": "video",
        "maxResults": max_results,
        "key": api_key,
    }
    response = requests.get("https://www.googleapis.com/youtube/v3/search", params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    rows = []
    for item in payload.get("items", []):
        snippet = item.get("snippet", {})
        video_id = item.get("id", {}).get("videoId")
        rows.append(
            {
                "source_system": "youtube",
                "query_text": query,
                "record_id": str(video_id),
                "title": snippet.get("title"),
                "url": f"https://www.youtube.com/watch?v={video_id}" if video_id else None,
                "description": snippet.get("description"),
                "published_at": snippet.get("publishedAt"),
                "source_payload": json.dumps(item, ensure_ascii=False),
            }
        )
    return pd.DataFrame(rows)


def landed_files_to_records(landing_dir: Path) -> pd.DataFrame:
    records = []
    if not landing_dir.exists():
        return pd.DataFrame(records)

    for file_path in sorted(landing_dir.glob("*")):
        suffix = file_path.suffix.lower()
        if suffix == ".csv":
            file_df = pd.read_csv(file_path)
        elif suffix == ".json":
            file_df = pd.read_json(file_path)
        else:
            continue

        for row_number, row in file_df.fillna("").astype(str).iterrows():
            payload = row.to_dict()
            records.append(
                {
                    "source_system": "landed_file",
                    "query_text": file_path.name,
                    "record_id": f"{file_path.stem}_{row_number}",
                    "title": payload.get("title") or payload.get("name") or payload.get("video_title") or file_path.stem,
                    "url": payload.get("url") or payload.get("link") or "",
                    "description": payload.get("description") or payload.get("summary") or json.dumps(payload, ensure_ascii=False),
                    "published_at": payload.get("published_at") or payload.get("release_date") or "",
                    "source_payload": json.dumps(payload, ensure_ascii=False),
                }
            )
    return pd.DataFrame(records)


In [0]:
batch_ts = datetime.now(timezone.utc).isoformat()
try:
    wiki_df = wikipedia_search(WIKI_QUERY, limit=12)
except requests.exceptions.HTTPError as e:
    print(f"Wikipedia API unavailable ({e.response.status_code}), continuing without Wikipedia data.")
    wiki_df = pd.DataFrame(columns=["source_system", "query_text", "record_id", "title", "url", "description", "published_at", "source_payload"])
yt_df = youtube_search(WIKI_QUERY, max_results=10, api_key=YOUTUBE_API_KEY)
landed_df = landed_files_to_records(LANDING_DIR)

source_frames = [df for df in [wiki_df, yt_df, landed_df] if not df.empty]
all_sources = pd.concat(source_frames, ignore_index=True) if source_frames else pd.DataFrame(columns=[
    "source_system", "query_text", "record_id", "title", "url", "description", "published_at", "source_payload"
])
all_sources["ingestion_ts"] = batch_ts

curated_df = all_sources.copy()
curated_df["entity_name"] = "Tom Cruise"
curated_df["content_type"] = curated_df["source_system"].map({
    "wikipedia": "biography_reference",
    "youtube": "video_reference",
    "landed_file": "file_enrichment",
}).fillna("other")

metrics_df = pd.DataFrame([
    {
        "ingestion_ts": batch_ts,
        "wiki_rows": int(len(wiki_df)),
        "youtube_rows": int(len(yt_df)),
        "landed_rows": int(len(landed_df)),
        "curated_rows": int(len(curated_df)),
        "landing_dir": str(LANDING_DIR),
        "youtube_enabled": bool(YOUTUBE_API_KEY),
    }
])

spark.createDataFrame(all_sources.fillna(""))\
    .createOrReplaceTempView("tc_dataset_sources_stage")
spark.createDataFrame(curated_df.fillna(""))\
    .createOrReplaceTempView("tc_dataset_stage")
spark.createDataFrame(metrics_df)\
    .createOrReplaceTempView("tc_dataset_ingest_metrics_stage")

spark.sql(f"CREATE OR REPLACE TABLE {RAW_SOURCE_TABLE} AS SELECT * FROM tc_dataset_sources_stage")
spark.sql(f"CREATE OR REPLACE TABLE {CURATED_TABLE} AS SELECT * FROM tc_dataset_stage")
spark.sql(f"CREATE OR REPLACE TABLE {INGEST_METRICS_TABLE} AS SELECT * FROM tc_dataset_ingest_metrics_stage")

print(f"Persisted source rows   : {len(all_sources)}")
print(f"Persisted curated rows  : {len(curated_df)}")
print(f"Persisted metrics table : {INGEST_METRICS_TABLE}")
display(spark.sql(f"SELECT source_system, COUNT(*) AS records FROM {RAW_SOURCE_TABLE} GROUP BY source_system ORDER BY source_system"))


Wikipedia API unavailable (403), continuing without Wikipedia data.
Persisted source rows   : 1
Persisted curated rows  : 1
Persisted metrics table : lakemeter_catalog.labuser3.tc_dataset_ingest_metrics


source_system,records
youtube,1


## ✅ Consumption Notes

* Run this notebook before executing the evaluation notebook if you want Tom Cruise source data available for retrieval context.
* Add new CSV or JSON files into [tc_landing](#folder-3047934714856900) and rerun this notebook to fold them into the curated dataset.
* Downstream table names:
  * `lakemeter_catalog.labuser3.tc_dataset_sources`
  * `lakemeter_catalog.labuser3.tc_dataset`
  * `lakemeter_catalog.labuser3.tc_dataset_ingest_metrics`
